# Pauli Propagation

In [3]:
import math
from functools import partial
import numpy as np
from matplotlib import pyplot as plt

from qiskit.quantum_info import Statevector, Operator
from quantum_simulation_recipe.plot_config import *
from quantum_simulation_recipe.trotter import pf, expH
from quantum_simulation_recipe.spin import Nearest_Neighbour_1d

from trotter_bounds import *
from pauli import *

fig_dir, data_dir = './figs', './data'
# plt.rcParams['font.family'] = 'sans-serif'
set_fontsize(linewidth=2.0)

## Key: Pauli rotation 

Let $R_G(\theta)[P]:=e^{iG\theta/2} P e^{-iG\theta/2}$,
then the evolution of Pauli operator with the Pauli rotation
$$R_G(\theta)[P]=P, \; [P,G]=0$$
$$R_G(\theta)[P]=\cos(\theta)P+\sin(\theta)P', \; P'=i[P,G]/2$$

In [4]:
XX = SparsePauliOp(['XX'],coeffs=[1.+0.j])
ZI = SparsePauliOp(['ZI'],coeffs=[1.+0.j])
ZZ = SparsePauliOp(['ZZ'],coeffs=[1.+0.j])

# print('XX:\n', XX.to_matrix())
# print('ZI:\n', ZI.to_matrix())

theta = math.pi/30
print('theta:', theta)  
expXX = expH(XX, theta)
# print('expXX:\n', expXX)
## set small number to zero by numpy setting
exact_result =  expXX.conj().T @ ZI.to_matrix() @ expXX
print('expXX[ZI]:\n', exact_result)

analytic_expression = pauli_rotation_evo(XX, theta, ZI)
# analytic_expression = math.cos(theta*2)*ZI + math.sin(theta*2)*1j*XX@ZI
print('Pauli rotation:', analytic_expression.simplify())

# Check the exact_result is equal to the analytic expression in matrix form
exact_result_analytic = analytic_expression.to_matrix()
print('exact_result_analytic:\n', exact_result_analytic)
print('exact_result == exact_result_analytic:', np.allclose(exact_result, exact_result_analytic))

pauli_weight_norm(analytic_expression)

theta: 0.10471975511965977
expXX[ZI]:
 [[ 0.978148+0.j        0.      +0.j        0.      +0.j
   0.      -0.207912j]
 [ 0.      +0.j        0.978148+0.j        0.      -0.207912j
   0.      +0.j      ]
 [ 0.      +0.j        0.      +0.207912j -0.978148+0.j
   0.      +0.j      ]
 [ 0.      +0.207912j  0.      +0.j        0.      +0.j
  -0.978148+0.j      ]]
Pauli rotation: SparsePauliOp(['ZI', 'YX'],
              coeffs=[0.978148+0.j, 0.207912-0.j])
exact_result_analytic:
 [[ 0.978148+0.j        0.      +0.j        0.      +0.j
   0.      -0.207912j]
 [ 0.      +0.j        0.978148+0.j        0.      -0.207912j
   0.      +0.j      ]
 [ 0.      +0.j        0.      +0.207912j -0.978148+0.j
   0.      +0.j      ]
 [ 0.      +0.207912j  0.      +0.j        0.      +0.j
  -0.978148+0.j      ]]
exact_result == exact_result_analytic: True


{1: 0.9567727288213006, 2: 0.043227271178699546}

## Compare norms

In [ ]:
n = 10
nnh = Nearest_Neighbour_1d(n, hy=1, Jx=2, pbc=False)

# np.linalg.norm(np.array([[1,1],[-1,1]]), ord='nuc')
print(nnh.ham)
print('Schatten 1-norm (trace/nuclear norm) =', np.linalg.norm(nnh.ham, ord='nuc'))
print('Schatten 2-norm (Frobenius/Hilbert norm) =', np.linalg.norm(nnh.ham, ord='fro'))
print('Schatten 2-norm =', np.sqrt(np.trace(nnh.ham@nnh.ham)))
print('Schatten infty-norm (operator norm) =', np.linalg.norm(nnh.ham, ord=2))
print('normalized 2-norm (Pauli 2-norm) =', np.sqrt(np.trace(nnh.ham@nnh.ham))*np.sqrt(1/2**n))
print('l1-norm (sum of absolute values) =', np.sum(np.abs(nnh.ham.coeffs)))
# print(np.trace(nnh.ham@nnh.ham))

SparsePauliOp(['IIIIIIIIXX', 'IIIIIIIXXI', 'IIIIIIXXII', 'IIIIIXXIII', 'IIIIXXIIII', 'IIIXXIIIII', 'IIXXIIIIII', 'IXXIIIIIII', 'XXIIIIIIII', 'IIIIIIIIIY', 'IIIIIIIIYI', 'IIIIIIIYII', 'IIIIIIYIII', 'IIIIIYIIII', 'IIIIYIIIII', 'IIIYIIIIII', 'IIYIIIIIII', 'IYIIIIIIII', 'YIIIIIIIII'],
              coeffs=[2.+0.j, 2.+0.j, 2.+0.j, 2.+0.j, 2.+0.j, 2.+0.j, 2.+0.j, 2.+0.j, 2.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j])
Schatten 1-norm (trace/nuclear norm) = 5607.974598662052
Schatten 2-norm (Frobenius/Hilbert norm) = 217.03455946000858
Schatten 2-norm = (217.03455946000858+0j)
Schatten infty-norm (operator norm) = 19.531007915854385
normalized 2-norm (Pauli 2-norm) = (6.782329983125268+0j)
l1-norm (sum of absolute values) = 28.0
